Import Dependencies

In [2]:
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,ToolMessage
from langchain_core.messages import convert_to_messages ,convert_to_openai_messages

from jinja2 import  Template
from typing import Literal,Dict,Any,Annotated,List
from IPython.display import Image,display
from operator import add
from openai import  OpenAI

import random
import ast
import inspect
import instructor
import json
from langchain_core.messages import AIMessage,ToolMessage,convert_to_openai_messages,HumanMessage,SystemMessage,BaseMessage
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct,Prefetch,FieldCondition,MatchText,FusionQuery,Document
import openai
import os
from langchain_openai import ChatOpenAI
from langsmith import traceable
from utils.tool import get_formatted_context,add_to_shopping_cart,remove_from_cart,get_shopping_cart,check_warehouse_availability,reserve_warehouse_items
from langgraph.checkpoint.postgres import PostgresSaver

from utils.utils import format_ai_message,parse_docstring_params,parse_function_definition

In [3]:
client=OpenAI()

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


## WareHouse Manager Agent

In [15]:
class RAGUsedContext(BaseModel):
    id:str=Field(description="The ID Of the item used answer the questions")
    description:str=Field(description="Short description of the item used to answer the Question")


class Toolcall(BaseModel):
    name:str
    arguments:dict

class FinalResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

class AgentProperties(BaseModel):
   iteration:int=0
   available_tools:List[dict[str,Any]]=[]
   tool_calls:List[Toolcall]=[]
   final_answer:bool=False

class Delegation(BaseModel):
    agent:str
    task:str

class CoordinatorAgentProperties(BaseModel):
    iteration:int=0
    final_answer:bool=False
    plan:List[Delegation]=[]
    next_agent:str=""
    

class State(BaseModel):
    messages:Annotated[List[Any],add]=[]
    question_relevant:bool=False 
    user_intent:str=""
    answer:str=""
    product_qa_agent:AgentProperties=Field(default_factory=CoordinatorAgentProperties)
    references:Annotated[List[RAGUsedContext],add]=[]
    shopping_cart_agent:AgentProperties=Field(default_factory=CoordinatorAgentProperties)
    coordinator_agent:CoordinatorAgentProperties=Field(default_factory=CoordinatorAgentProperties)
    warehouse_manager_agent:AgentProperties=Field(default_factory=CoordinatorAgentProperties)
    user_id:str=""
    cart_id:str=""


In [8]:
class ProductQnAgentResponse(BaseModel):
    answer:str
    tool_calls:List[Toolcall]=Field(default_factory=list)
    final_answer: bool = Field(description="True if you have all the information needed to provide a complete answer, False otherwise.")
    references: List[RAGUsedContext] = Field(default_factory=list, description="List of items used to answer the question")

class ShoppingCartAgentResponse(BaseModel):
    answer:str
    tool_calls:List[Toolcall]=Field(default_factory=list)
    final_answer: bool = Field(description="True if you have all the information needed to provide a complete answer, False otherwise.")
    references: List[RAGUsedContext] = Field(default_factory=list, description="List of items used to answer the question")


WareHouseManger_agent


In [16]:
class ProductQnAgentResponse(BaseModel):
    answer:str
    tool_calls:List[Toolcall]=Field(default_factory=list)
    final_answer: bool = Field(description="True if you have all the information needed to provide a complete answer, False otherwise.")
    references: List[RAGUsedContext] = Field(default_factory=list, description="List of items used to answer the question")

class WarehouseAgentResponse(BaseModel):
    answer:str
    tool_calls:List[Toolcall]=Field(default_factory=list)
    final_answer: bool = Field(description="True if you have all the information needed to provide a complete answer, False otherwise.")
    references: List[RAGUsedContext] = Field(default_factory=list, description="List of items used to answer the question")


## Coordinator Agent

In [33]:
class Delegation(BaseModel):
    agent:str
    task:str

class CoordinatorAgent(BaseModel):
    next_agent: str
    plan: List[Delegation]
    final_answer: bool = Field(
        default=False, 
        description="Set to True if you are answering the user directly. Set to False if you are delegating to another agent."
    )
    answer: str


In [34]:
def getconvesationhistory(messages)->[List]:
    conversation=[]
    for message in messages:
        if type(message).__name__ == "HumanMessage":
            conversation.append(convert_to_openai_messages(message))
        elif isinstance(message, dict) and message.get("role") == "user":
            conversation.append(message)
        elif type(message).__name__ == "AIMessage":
            if message.content: 
                clean_msg = AIMessage(content=message.content)
                conversation.append(convert_to_openai_messages(clean_msg))
        elif isinstance(message, dict) and message.get("role") == "assistant":
            if message.get("content"):
                conversation.append({"role": "assistant", "content": message["content"]})
        elif type(message).__name__ == "ToolMessage":
            clean_msg = HumanMessage(content=f"Tool [{message.name}] Output:\n{message.content}")
            conversation.append(convert_to_openai_messages(clean_msg))
        elif isinstance(message, dict) and message.get("role") == "tool":
            conversation.append({"role": "user", "content": f"Tool [{message.get('name')}] Output:\n{message.get('content')}"})
        
    return conversation


In [35]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={"ls_provider":"openai"}
)
def coordinator_agent(state):
    
    prompt_template="""  You are a Coordinator Agent as part of a shopping assistant.

    ## Instructions

    - Your role is to create plans for solving user queries and delegate the tasks accordingly.
    - You will be given a conversation history, your task is to create a plan for solving the user's query.
    - After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
    - Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
    - If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
    - Do not route to any agent if the user's query needs clarification or is irrelevant. Do it yourself.

    ## Available Agents

    - product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
    - shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
    - warehouse_manager_agent: The user is asking items from the Warehouse

    ## Examples

    Question: "Do you have running shoes under $100?"
    Next agent: product_qna_agent

    Question: "Can you list the items in my cart?"
    Next agent: shopping_cart_agent
    """
    template = Template(prompt_template)
    prompt = template.render()

    messages = state.messages
    conversation = []

    conversation = getconvesationhistory(messages=messages)    
    client = instructor.from_openai(OpenAI())

    response,raw_response=client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        response_model=CoordinatorAgent,
        messages=[
            {
                "role":"system",

                 "content":prompt,
            },
            *conversation
        ],
        temperature=0.5,
    )

    if response.final_answer:
        ai_message=[AIMessage(content=response.answer)]
    else:
        ai_message=[]


    return {
        "messages":ai_message,
        "user_intent":"",
        "answer":response.answer,
        "coordinator_agent":{
            "iteration":state.coordinator_agent.iteration+1,
            "final_answer":response.final_answer,
            "next_agent":response.next_agent,
            "plan":[item.model_dump() for item in response.plan]

        }
    }


### Coordinator agent Edge

In [36]:
def coordinator_agent_edge(state: State) -> str:
    if state.coordinator_agent.final_answer:
        return "end"
    elif state.coordinator_agent.iteration > 4:
        return "end"
    elif state.coordinator_agent.next_agent == "product_qna_agent":
        return "product_qna_agent"
    elif state.coordinator_agent.next_agent == "shopping_cart_agent":
        return "shopping_cart_agent"
    else:
        return "end"

In [37]:
initial_state=State(
    messages=[{"role":"user","content":"What is the Weather today"}]
)

In [38]:
answer=coordinator_agent(initial_state)

In [39]:
answer

{'messages': [AIMessage(content='I am here to assist with shopping-related queries. For weather information, you may want to check a weather website or app.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'user_intent': '',
 'answer': 'I am here to assist with shopping-related queries. For weather information, you may want to check a weather website or app.',
 'coordinator_agent': {'iteration': 1,
  'final_answer': True,
  'next_agent': '',
  'plan': []}}

In [40]:
initial_state=State(
    messages=[{"role":"user","content":"Can You give me some Earphones"}]
)

In [41]:
answer=coordinator_agent(initial_state)

In [42]:
answer

{'messages': [],
 'user_intent': '',
 'answer': 'I will find some earphones for you. Please wait a moment.',
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'next_agent': 'product_qna_agent',
  'plan': [{'agent': 'product_qna_agent',
    'task': 'Provide a list of available earphones with their specifications and prices.'}]}}

## Coordinator Evaluation Dataset and Runner

Here we define our evaluation dataset containing test queries spanning all available agents (`product_qna_agent`, `shopping_cart_agent`, and `warehouse_manager_agent`) as well as irrelevant/out-of-scope requests. We then run the coordinator agent on this dataset and record performance metrics.

In [49]:
import json
with open("coordinator_eval_dataset.json", "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)
print(f"Loaded evaluation dataset with {len(eval_dataset)} test cases.")

Loaded evaluation dataset with 32 test cases.


In [51]:
eval_dataset[4]

{'input': {'messages': [{'role': 'user',
    'content': 'Find me some wireless earbuds that have active noise cancellation.'}]},
 'outputs': {'next_agent': 'product_qna_agent',
  'coordinator_final_answer': False}}

In [ ]:
import pandas as pd
from tqdm import tqdm

results = []

print("Running evaluation...")
for i, case in enumerate(tqdm(eval_dataset)):
    messages_input = case["input"]["messages"]
    expected_agent = case["outputs"]["next_agent"]
    expected_final = case["outputs"]["coordinator_final_answer"]
    
    # Initialize state with messages list
    state = State(messages=messages_input)
    
    try:
        # Execute coordinator_agent
        output = coordinator_agent(state)
        
        pred_agent = output["coordinator_agent"]["next_agent"]
        pred_final = output["coordinator_agent"]["final_answer"]
        
        agent_match = (pred_agent == expected_agent)
        final_match = (pred_final == expected_final)
        
        results.append({
            "index": i + 1,
            "query": messages_input[-1]["content"] if messages_input else "",
            "expected_agent": expected_agent,
            "pred_agent": pred_agent,
            "agent_correct": agent_match,
            "expected_final": expected_final,
            "pred_final": pred_final,
            "final_correct": final_match,
            "all_correct": agent_match and final_match,
            "answer": output["answer"]
        })
    except Exception as e:
        results.append({
            "index": i + 1,
            "query": messages_input[-1]["content"] if messages_input else "",
            "expected_agent": expected_agent,
            "pred_agent": f"ERROR: {str(e)}",
            "agent_correct": False,
            "expected_final": expected_final,
            "pred_final": None,
            "final_correct": False,
            "all_correct": False,
            "answer": ""
        })

df_results = pd.DataFrame(results)

# Calculate metrics
agent_accuracy = df_results["agent_correct"].mean() * 100
final_accuracy = df_results["final_correct"].mean() * 100
overall_accuracy = df_results["all_correct"].mean() * 100

print("\n--- EVALUATION SUMMARY ---")
print(f"Total Test Cases: {len(df_results)}")
print(f"Agent Routing Accuracy: {agent_accuracy:.2f}%")
print(f"Final Answer Flag Accuracy: {final_accuracy:.2f}%")
print(f"Overall Exact Match Accuracy: {overall_accuracy:.2f}%")

# Display incorrect predictions
incorrect_runs = df_results[~df_results["all_correct"]]
if not incorrect_runs.empty:
    print(f"\nFound {len(incorrect_runs)} mismatched test cases:")
    display(incorrect_runs[["query", "expected_agent", "pred_agent", "expected_final", "pred_final", "answer"]])
else:
    print("\nAll test cases passed successfully!")


In [54]:
from langsmith import Client
import os
client=Client(api_key=os.getenv("LANGSMITH_API_KEY"))
dataset_name="coordinatoor_eval_dataset"
dataset=client.create_dataset(
    dataset_name=dataset_name,
    description="Coordinator Agent Evaluation Dtasets For the first routing step"
)

In [ ]:
for item in eval_dataset:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"messages":item["input"]["messages"]},
        outputs={
            "next_agent":item["outputs"]["next_agent"],
            "coordinator_final_answer":item["outputs"]["coordinator_final_answer"]
        }
    )

In [58]:
def next_agent_evaluator(run,example):
    next_agent_match=run.outputs["coordinator_agent"]["next_agent"]==example.outputs["next_agent"]
    final_answer_match=run.outputs["coordinator_agent"]["final_answer"]==example.outputs["coordinator_final_answer"]

    return next_agent_match and  final_answer_match

In [63]:
results = client.evaluate(
    lambda x: coordinator_agent(
        State(messages=x['messages']) # <-- State and coordinator_agent are correctly closed here
    ),
    data="coordinatoor_eval_dataset", # <-- Now correctly passed to client.evaluate
    evaluators=[
        next_agent_evaluator
    ],
    experiment_prefix="coordinatoor_eval_dataset"
)


View the evaluation results for experiment: 'coordinatoor_eval_dataset-a1055c5a' at:
https://smith.langchain.com/o/0446773b-8ef0-431f-8945-5ad0eb3e1d41/datasets/a67bf718-97d7-4fb4-8a4c-05933cfdb089/compare?selectedSessions=09453c24-c279-41ee-8fe4-fa6ef35fe7bd




38it [01:08,  1.80s/it]
